# Experiment 7 — LightGBM × 5 Imbalance Techniques

LightGBM is a gradient boosting framework using histogram-based learning.
Faster than XGBoost on large datasets. Native support for imbalanced data via `is_unbalance` and `scale_pos_weight`.

**LightGBM advantages over XGBoost:**
- Faster training (histogram-based splits)
- Lower memory usage
- Native focal loss via custom or sigmoid objective

**5 Techniques:** Baseline, Class Weights, SMOTE, Focal Loss, Threshold Optimization

**Prerequisite:** Run `00_dataset_prep.ipynb` first.

In [1]:
import os, sys, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

sys.path.insert(0, os.path.abspath('..'))
from utils.metrics import compute_all_metrics, print_metrics

DATA_DIR    = os.path.join('..', 'data')
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Fixed LightGBM params — equivalent depth/trees to XGBoost for fair comparison
LGB_PARAMS = dict(n_estimators=500, max_depth=6, learning_rate=0.1,
                  random_state=42, verbose=-1, n_jobs=4)

SMOTE_CAP  = 500_000
THRESHOLDS = np.arange(0.1, 0.91, 0.01)
print("Experiment 7 — LightGBM × 5 Techniques")

Experiment 7 — LightGBM × 5 Techniques


In [2]:
def run_lgb_technique(X_train, y_train, X_test, y_test, technique, v):
    """Run one LightGBM technique and return metrics + probs."""
    t0 = time.time()

    if technique == 'baseline':
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        train_time = time.time() - t0
        return compute_all_metrics(y_test, probs, preds, train_time), probs

    elif technique == 'class_weights':
        n_bg = (y_train==0).sum(); n_sig = (y_train==1).sum()
        spw  = n_bg / n_sig
        model = lgb.LGBMClassifier(**LGB_PARAMS, scale_pos_weight=spw)
        model.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        train_time = time.time() - t0
        return compute_all_metrics(y_test, probs, preds, train_time), probs

    elif technique == 'smote':
        if v == 'A' and len(X_train) > SMOTE_CAP:
            idx = np.random.RandomState(42).choice(len(X_train), SMOTE_CAP, replace=False)
            X_s, y_s = X_train[idx], y_train[idx]
        else:
            X_s, y_s = X_train, y_train
        X_res, y_res = SMOTE(random_state=42).fit_resample(X_s, y_s)
        smote_time = time.time() - t0
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(X_res, y_res)
        probs = model.predict_proba(X_test)[:,1]
        preds = model.predict(X_test)
        train_time = time.time() - t0 + smote_time
        return compute_all_metrics(y_test, probs, preds, train_time), probs

    elif technique == 'focal_loss':
        # LightGBM custom focal loss
        def lgb_focal_loss(y_true, y_pred):
            gamma = 2.0
            p = 1.0 / (1.0 + np.exp(-y_pred))
            grad = -(y_true * (1 - p)**gamma * (1 - p) - (1 - y_true) * p**gamma * p)
            hess = p * (1 - p)
            return grad, hess

        model = lgb.LGBMClassifier(**LGB_PARAMS, objective=lgb_focal_loss)
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model.fit(X_tr, y_tr)
        val_raw = model.predict(X_val, raw_score=True)
        val_probs = 1.0 / (1.0 + np.exp(-val_raw))
        f1s   = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                 for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        test_raw = model.predict(X_test, raw_score=True)
        probs  = 1.0 / (1.0 + np.exp(-test_raw))
        preds  = (probs >= best_t).astype(int)
        train_time = time.time() - t0
        m = compute_all_metrics(y_test, probs, preds, train_time)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

    elif technique == 'threshold':
        X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2,
                                                      random_state=42, stratify=y_train)
        model = lgb.LGBMClassifier(**LGB_PARAMS)
        model.fit(X_tr, y_tr)
        val_probs = model.predict_proba(X_val)[:,1]
        f1s    = [f1_score(y_val, (val_probs>=t).astype(int), pos_label=1, zero_division=0)
                  for t in THRESHOLDS]
        best_t = THRESHOLDS[int(np.argmax(f1s))]
        probs  = model.predict_proba(X_test)[:,1]
        preds  = (probs >= best_t).astype(int)
        train_time = time.time() - t0
        m = compute_all_metrics(y_test, probs, preds, train_time)
        m['Best_Threshold'] = round(best_t, 2)
        return m, probs

In [3]:
techniques = ['baseline','class_weights','smote','focal_loss','threshold']
tech_names = {'baseline':'Baseline','class_weights':'Class Weights',
              'smote':'SMOTE','focal_loss':'Focal Loss','threshold':'Threshold Opt.'}
all_results = []

for v in ['A','B','C']:
    print(f"\n[Exp7-LGB] Version {v}")
    try:
        train = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_train.csv'))
        test  = pd.read_csv(os.path.join(DATA_DIR, f'version_{v}_test.csv'))
    except FileNotFoundError:
        print(f"ERROR: Run 00_dataset_prep.ipynb first."); raise

    X_train = train.drop('label',axis=1).values; y_train = train['label'].values
    X_test  = test.drop('label',axis=1).values;  y_test  = test['label'].values

    for tech in techniques:
        print(f"  [{tech}] running...")
        metrics, probs = run_lgb_technique(X_train, y_train, X_test, y_test, tech, v)
        metrics['Version']   = v
        metrics['Technique'] = tech_names[tech]
        all_results.append(metrics)
        np.save(os.path.join(RESULTS_DIR, f'probs_lgb_{tech}_version_{v}.npy'), probs)
        print_metrics(metrics, label=f"LGB-{tech} V{v}")

print("\n[Exp7-LGB] All complete.")


[Exp7-LGB] Version A
  [baseline] running...
[LGB-baseline VA] AUC=0.8191 | F1=0.7542 | Signal_Sig=178.3639 | Train_Time=9.77s
  [class_weights] running...
[LGB-class_weights VA] AUC=0.8188 | F1=0.7464 | Signal_Sig=178.4095 | Train_Time=10.63s
  [smote] running...
[LGB-smote VA] AUC=0.8175 | F1=0.7473 | Signal_Sig=178.3332 | Train_Time=77.41s
  [focal_loss] running...
[LGB-focal_loss VA] AUC=0.8157 | F1=0.7652 | Signal_Sig=170.5625 | Train_Time=20.62s
  [threshold] running...
[LGB-threshold VA] AUC=0.8185 | F1=0.7670 | Signal_Sig=178.2002 | Train_Time=9.69s

[Exp7-LGB] Version B
  [baseline] running...
[LGB-baseline VB] AUC=0.8107 | F1=0.2322 | Signal_Sig=25.9795 | Train_Time=5.31s
  [class_weights] running...
[LGB-class_weights VB] AUC=0.8088 | F1=0.3603 | Signal_Sig=25.4985 | Train_Time=5.43s
  [smote] running...
[LGB-smote VB] AUC=0.7619 | F1=0.3431 | Signal_Sig=24.3623 | Train_Time=11.86s
  [focal_loss] running...
[LGB-focal_loss VB] AUC=0.8110 | F1=0.3990 | Signal_Sig=26.8787 | T

In [4]:
cols = ['Version','Technique','AUC','F1','Signal_Significance','Train_Time_sec']
results_df = pd.DataFrame(all_results)
results_df = results_df[[c for c in cols if c in results_df.columns]]
save_path  = os.path.join(RESULTS_DIR, 'experiment7_lightgbm.csv')
results_df.to_csv(save_path, index=False)
print(results_df.to_string(index=False))
print(f"\n✓ Saved → {save_path}")

Version      Technique      AUC       F1  Signal_Significance  Train_Time_sec
      A       Baseline 0.819133 0.754205           178.363924            9.77
      A  Class Weights 0.818843 0.746366           178.409460           10.63
      A          SMOTE 0.817548 0.747251           178.333232           77.41
      A     Focal Loss 0.815676 0.765225           170.562487           20.62
      A Threshold Opt. 0.818502 0.767032           178.200167            9.69
      B       Baseline 0.810736 0.232184            25.979501            5.31
      B  Class Weights 0.808826 0.360272            25.498507            5.43
      B          SMOTE 0.761892 0.343056            24.362283           11.86
      B     Focal Loss 0.811015 0.398991            26.878725           12.63
      B Threshold Opt. 0.809303 0.398556            25.991293            5.30
      C       Baseline 0.774944 0.024768             3.137858            4.80
      C  Class Weights 0.756310 0.129771             5.342169   